# E-Commerce Sales & Profit Analysis

---

## Section 1 — Project Overview

### Business Problem
The business wants to understand:
- How much revenue is being generated?
- How profitable is the business?
- Which products generate the most revenue and profit?
- Which customer segments are most valuable?
- Which regions/states perform best?
- How are sales changing over time?
- How does discount affect profit?
- Where are opportunities to improve profitability?

### Objectives
1. Analyze sales performance across multiple dimensions
2. Identify profitability patterns and loss-making transactions
3. Understand customer behavior and segment performance
4. Evaluate the impact of discounts on profitability
5. Provide actionable business recommendations

### Dataset Source
**Dataset:** Sample Superstore Dataset  
**Original Row Count:** 9,994 rows  
**Final Cleaned Row Count:** TBD (after cleaning)

### Key Business Questions
1. What is total revenue and profit?
2. Which categories and products perform best?
3. Which customer segments are most valuable?
4. How do discounts impact profitability?
5. Which regions and states generate the most revenue?
6. What are the monthly sales and profit trends?
7. Which products are loss-making?
8. What is the profit margin across different dimensions?

---

## Section - 2 Load the raw dataset

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---

## Section 3 — Load Dataset

In [2]:
# Load the raw dataset
df = pd.read_csv('../data/raw/ecommerce_raw.csv', encoding='latin1')

# Display basic information
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nDataset memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

DATASET OVERVIEW
Number of rows: 9,994
Number of columns: 21

Dataset memory usage: 9.21 MB


In [3]:
# Display first 5 rows
print("\nFirst 5 rows:")
print("="*60)
df.head()


First 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


In [4]:
# Display data types
print("\nData Types:")
print("="*60)
df.dtypes


Data Types:


Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [5]:
# Display column names
print("\nColumn Names:")
print("="*60)
print(df.columns.tolist())


Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


---

## Section 4 — Data Quality Assessment

In [6]:
print("="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

# Missing values
print("\n1. Missing Values:")
print("-"*60)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found")

DATA QUALITY ASSESSMENT

1. Missing Values:
------------------------------------------------------------
No missing values found


In [7]:
# Duplicate rows
print("\n2. Duplicate Rows:")
print("-"*60)
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")


2. Duplicate Rows:
------------------------------------------------------------
Number of duplicate rows: 0


In [8]:
# Duplicate Order IDs
print("\n3. Duplicate Order IDs:")
print("-"*60)
duplicate_orders = df['Order ID'].duplicated().sum()
print(f"Number of duplicate Order IDs: {duplicate_orders}")
print(f"Note: Duplicate Order IDs are expected as one order can have multiple products")


3. Duplicate Order IDs:
------------------------------------------------------------
Number of duplicate Order IDs: 4985
Note: Duplicate Order IDs are expected as one order can have multiple products


In [9]:
# Check date columns
print("\n4. Date Column Validation:")
print("-"*60)
print(f"Order Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"Ship Date range: {df['Ship Date'].min()} to {df['Ship Date'].max()}")


4. Date Column Validation:
------------------------------------------------------------
Order Date range: 1/1/2017 to 9/9/2017
Ship Date range: 1/1/2015 to 9/9/2017


In [10]:
# Negative values in numeric columns
print("\n5. Negative Values Check:")
print("-"*60)
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")


5. Negative Values Check:
------------------------------------------------------------
Sales: 0 negative values
Quantity: 0 negative values
Discount: 0 negative values
Profit: 1871 negative values


In [11]:
# Zero values in numeric columns
print("\n6. Zero Values Check:")
print("-"*60)
for col in numeric_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count} zero values")


6. Zero Values Check:
------------------------------------------------------------
Sales: 0 zero values
Quantity: 0 zero values
Discount: 4798 zero values
Profit: 65 zero values


In [12]:
# Unique values in categorical columns
print("\n7. Unique Values in Categorical Columns:")
print("-"*60)
categorical_cols = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} unique values")


7. Unique Values in Categorical Columns:
------------------------------------------------------------
Ship Mode: 4 unique values
Segment: 3 unique values
Region: 4 unique values
Category: 3 unique values
Sub-Category: 17 unique values


In [13]:
# Statistical summary
print("\n8. Statistical Summary:")
print("-"*60)
df[numeric_cols].describe()


8. Statistical Summary:
------------------------------------------------------------


,Sales,Quantity,Discount,Profit
count,9994.00,9994.00,9994.00,9994.00
mean,229.86,3.79,0.16,28.66
std,623.25,2.23,0.21,234.26
min,0.44,1.00,0.00,-6599.98
25%,17.28,2.00,0.00,1.73
50%,54.49,3.00,0.20,8.67
75%,209.94,5.00,0.20,29.36
max,22638.48,14.00,0.80,8399.98


In [14]:
# Data Quality Summary Table
print("\n9. Data Quality Summary Table:")
print("-"*60)
quality_summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Missing': df.isnull().sum().values,
    'Unique': df.nunique().values,
    'Issues': [
        'Negative values present' if col in ['Profit', 'Sales'] and (df[col] < 0).sum() > 0 else
        'Zero values present' if col in ['Sales', 'Quantity'] and (df[col] == 0).sum() > 0 else
        'None'
        for col in df.columns
    ]
})
quality_summary


9. Data Quality Summary Table:
------------------------------------------------------------


,Column,Data Type,Missing,Unique,Issues
0,Row ID,int64,0,9994,None
1,Order ID,str,0,5009,None
2,Order Date,str,0,1237,None
3,Ship Date,str,0,1334,None
4,Ship Mode,str,0,4,None
5,Customer ID,str,0,793,None
6,Customer Name,str,0,793,None
7,Segment,str,0,3,None
8,Country,str,0,1,None
9,City,str,0,531,None


---

## Section 5 — Data Cleaning

In [15]:
print("="*60)
print("DATA CLEANING")
print("="*60)

# Create a copy for cleaning
df_clean = df.copy()
print(f"Original dataset shape: {df_clean.shape}")

DATA CLEANING
Original dataset shape: (9994, 21)


In [16]:
# 1. Convert date columns
print("\n1. Converting date columns...")
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'], format='%m/%d/%Y')
df_clean['Ship Date'] = pd.to_datetime(df_clean['Ship Date'], format='%m/%d/%Y')
print("Date columns converted successfully")


1. Converting date columns...
Date columns converted successfully


In [17]:
# 2. Standardize column names (replace spaces with underscores)
print("\n2. Standardizing column names...")
df_clean.columns = df_clean.columns.str.replace(' ', '_')
print("Column names standardized")
print(df_clean.columns.tolist())


2. Standardizing column names...
Column names standardized
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [18]:
# 3. Check for and remove completely duplicate rows (if any)
print("\n3. Checking for completely duplicate rows...")
before_dedup = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after_dedup = df_clean.shape[0]
print(f"Rows before deduplication: {before_dedup}")
print(f"Rows after deduplication: {after_dedup}")
print(f"Duplicate rows removed: {before_dedup - after_dedup}")


3. Checking for completely duplicate rows...
Rows before deduplication: 9994
Rows after deduplication: 9994
Duplicate rows removed: 0


In [19]:
# 4. Create derived features
print("\n4. Creating derived features...")

# Date-based features
df_clean['Year'] = df_clean['Order_Date'].dt.year
df_clean['Quarter'] = df_clean['Order_Date'].dt.quarter
df_clean['Month'] = df_clean['Order_Date'].dt.month
df_clean['Month_Name'] = df_clean['Order_Date'].dt.month_name()
df_clean['Year_Month'] = df_clean['Order_Date'].dt.to_period('M')
df_clean['Week'] = df_clean['Order_Date'].dt.isocalendar().week
df_clean['Day_of_Week'] = df_clean['Order_Date'].dt.day_name()

# Order-to-Ship days
df_clean['Order_to_Ship_Days'] = (df_clean['Ship_Date'] - df_clean['Order_Date']).dt.days

# Profit Margin
df_clean['Profit_Margin'] = (df_clean['Profit'] / df_clean['Sales']) * 100

# Sales per Quantity
df_clean['Sales_per_Quantity'] = df_clean['Sales'] / df_clean['Quantity']

# Discount Band
def get_discount_band(discount):
    if discount == 0:
        return '0%'
    elif discount <= 0.10:
        return '1-10%'
    elif discount <= 0.20:
        return '11-20%'
    elif discount <= 0.30:
        return '21-30%'
    else:
        return '30%+'

df_clean['Discount_Band'] = df_clean['Discount'].apply(get_discount_band)

print("Derived features created successfully")
print(f"New columns added: Year, Quarter, Month, Month_Name, Year_Month, Week, Day_of_Week, Order_to_Ship_Days, Profit_Margin, Sales_per_Quantity, Discount_Band")


4. Creating derived features...
Derived features created successfully
New columns added: Year, Quarter, Month, Month_Name, Year_Month, Week, Day_of_Week, Order_to_Ship_Days, Profit_Margin, Sales_per_Quantity, Discount_Band


In [20]:
# 5. Handle infinite values in Profit_Margin (caused by zero Sales)
print("\n5. Handling infinite values in Profit_Margin...")
infinite_count = np.isinf(df_clean['Profit_Margin']).sum()
print(f"Infinite values in Profit_Margin: {infinite_count}")
df_clean['Profit_Margin'] = df_clean['Profit_Margin'].replace([np.inf, -np.inf], 0)
print("Infinite values replaced with 0")


5. Handling infinite values in Profit_Margin...
Infinite values in Profit_Margin: 0
Infinite values replaced with 0


In [21]:
# 6. Investigate outliers in Sales and Profit
print("\n6. Outlier Investigation:")
print("-"*60)

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers

sales_outliers = detect_outliers_iqr(df_clean, 'Sales')
profit_outliers = detect_outliers_iqr(df_clean, 'Profit')

print(f"Sales outliers (IQR method): {len(sales_outliers)} ({len(sales_outliers)/len(df_clean)*100:.2f}%)")
print(f"Profit outliers (IQR method): {len(profit_outliers)} ({len(profit_outliers)/len(df_clean)*100:.2f}%)")
print("\nDecision: Outliers will NOT be removed as they represent legitimate high-value transactions and loss-making orders that are important for business analysis.")


6. Outlier Investigation:
------------------------------------------------------------
Sales outliers (IQR method): 1167 (11.68%)
Profit outliers (IQR method): 1881 (18.82%)

Decision: Outliers will NOT be removed as they represent legitimate high-value transactions and loss-making orders that are important for business analysis.


In [22]:
# 7. Final dataset shape after cleaning
print("\n7. Final Dataset Summary:")
print("-"*60)
print(f"Original row count: {df.shape[0]:,}")
print(f"Final row count: {df_clean.shape[0]:,}")
print(f"Original column count: {df.shape[1]}")
print(f"Final column count: {df_clean.shape[1]}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]:,}")


7. Final Dataset Summary:
------------------------------------------------------------
Original row count: 9,994
Final row count: 9,994
Original column count: 21
Final column count: 32
Rows removed: 0


In [23]:
# Display cleaned dataset sample
print("\nSample of cleaned dataset:")
df_clean.head()


Sample of cleaned dataset:


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Year,Quarter,Month,Month_Name,Year_Month,Week,Day_of_Week,Order_to_Ship_Days,Profit_Margin,Sales_per_Quantity,Discount_Band
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,2016,4,11,November,2016-11,45,Tuesday,3,16.00,130.98,0%
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,2016,4,11,November,2016-11,45,Tuesday,3,30.00,243.98,0%
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,2016,2,6,June,2016-06,23,Sunday,4,47.00,7.31,0%
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,2015,4,10,October,2015-10,41,Sunday,7,-40.00,191.52,30%+
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52,2015,4,10,October,2015-10,41,Sunday,7,11.25,11.18,11-20%


---

## Section 6 — Exploratory Data Analysis

### 6.1 Sales Analysis

In [24]:
print("="*60)
print("SALES ANALYSIS")
print("="*60)

# Total Sales
total_sales = df_clean['Sales'].sum()
print(f"\nTotal Sales: ${total_sales:,.2f}")

SALES ANALYSIS

Total Sales: $2,297,200.86


In [25]:
# Sales by Year
print("\nSales by Year:")
print("-"*60)
sales_by_year = df_clean.groupby('Year')['Sales'].sum().sort_index()
print(sales_by_year)


Sales by Year:
------------------------------------------------------------
Year
2014   484247.50
2015   470532.51
2016   609205.60
2017   733215.26
Name: Sales, dtype: float64


In [26]:
# Sales by Category
print("\nSales by Category:")
print("-"*60)
sales_by_category = df_clean.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print(sales_by_category)


Sales by Category:
------------------------------------------------------------
Category
Technology        836154.03
Furniture         741999.80
Office Supplies   719047.03
Name: Sales, dtype: float64


In [27]:
# Sales by Region
print("\nSales by Region:")
print("-"*60)
sales_by_region = df_clean.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(sales_by_region)


Sales by Region:
------------------------------------------------------------
Region
West      725457.82
East      678781.24
Central   501239.89
South     391721.91
Name: Sales, dtype: float64


In [28]:
# Sales by Segment
print("\nSales by Customer Segment:")
print("-"*60)
sales_by_segment = df_clean.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
print(sales_by_segment)


Sales by Customer Segment:
------------------------------------------------------------
Segment
Consumer      1161401.34
Corporate      706146.37
Home Office    429653.15
Name: Sales, dtype: float64


In [29]:
# Top 10 States by Sales
print("\nTop 10 States by Sales:")
print("-"*60)
sales_by_state = df_clean.groupby('State')['Sales'].sum().sort_values(ascending=False).head(10)
print(sales_by_state)


Top 10 States by Sales:
------------------------------------------------------------
State
California     457687.63
New York       310876.27
Texas          170188.05
Washington     138641.27
Pennsylvania   116511.91
Florida         89473.71
Illinois        80166.10
Ohio            78258.14
Michigan        76269.61
Virginia        70636.72
Name: Sales, dtype: float64


### 6.2 Profit Analysis

In [30]:
print("="*60)
print("PROFIT ANALYSIS")
print("="*60)

# Total Profit
total_profit = df_clean['Profit'].sum()
print(f"\nTotal Profit: ${total_profit:,.2f}")

PROFIT ANALYSIS

Total Profit: $286,397.02


In [31]:
# Overall Profit Margin
overall_profit_margin = (total_profit / total_sales) * 100
print(f"Overall Profit Margin: {overall_profit_margin:.2f}%")

Overall Profit Margin: 12.47%


In [32]:
# Profit by Category
print("\nProfit by Category:")
print("-"*60)
profit_by_category = df_clean.groupby('Category')['Profit'].sum().sort_values(ascending=False)
print(profit_by_category)


Profit by Category:
------------------------------------------------------------
Category
Technology        145454.95
Office Supplies   122490.80
Furniture          18451.27
Name: Profit, dtype: float64


In [33]:
# Profit by Region
print("\nProfit by Region:")
print("-"*60)
profit_by_region = df_clean.groupby('Region')['Profit'].sum().sort_values(ascending=False)
print(profit_by_region)


Profit by Region:
------------------------------------------------------------
Region
West      108418.45
East       91522.78
South      46749.43
Central    39706.36
Name: Profit, dtype: float64


In [34]:
# Loss-making transactions
print("\nLoss-Making Analysis:")
print("-"*60)
loss_transactions = df_clean[df_clean['Profit'] < 0]
print(f"Number of loss-making transactions: {len(loss_transactions)}")
print(f"Percentage of transactions: {len(loss_transactions)/len(df_clean)*100:.2f}%")
print(f"Total loss: ${loss_transactions['Profit'].sum():,.2f}")


Loss-Making Analysis:
------------------------------------------------------------
Number of loss-making transactions: 1871
Percentage of transactions: 18.72%
Total loss: $-156,131.29


In [35]:
# Top 10 Profitable Products
print("\nTop 10 Products by Profit:")
print("-"*60)
top_products_profit = df_clean.groupby('Product_Name')['Profit'].sum().sort_values(ascending=False).head(10)
print(top_products_profit)


Top 10 Products by Profit:
------------------------------------------------------------
Product_Name
Canon imageCLASS 2200 Advanced Copier                                         25199.93
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind    7753.04
Hewlett Packard LaserJet 3310 Copier                                           6983.88
Canon PC1060 Personal Laser Copier                                             4570.93
HP Designjet T520 Inkjet Large Format Printer - 24" Color                      4094.98
Ativa V4110MDD Micro-Cut Shredder                                              3772.95
3D Systems Cube Printer, 2nd Generation, Magenta                               3717.97
Plantronics Savi W720 Multi-Device Wireless Headset System                     3696.28
Ibico EPK-21 Electric Binding System                                           3345.28
Zebra ZM400 Thermal Label Printer                                              3343.54
Name: Profit, dtype: float64

In [36]:
# Bottom 10 Products by Profit (Loss-making)
print("\nBottom 10 Products by Profit (Loss-making):")
print("-"*60)
bottom_products_profit = df_clean.groupby('Product_Name')['Profit'].sum().sort_values(ascending=True).head(10)
print(bottom_products_profit)


Bottom 10 Products by Profit (Loss-making):
------------------------------------------------------------
Product_Name
Cubify CubeX 3D Printer Double Head Print                           -8879.97
Lexmark MX611dhe Monochrome Laser Printer                           -4589.97
Cubify CubeX 3D Printer Triple Head Print                           -3839.99
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2876.12
Bush Advantage Collection Racetrack Conference Table                -1934.40
GBC DocuBind P400 Electric Binding System                           -1878.17
Cisco TelePresence System EX90 Videoconferencing Unit               -1811.08
Martin Yale Chadless Opener Electric Letter Opener                  -1299.18
Balt Solid Wood Round Tables                                        -1201.06
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1148.44
Name: Profit, dtype: float64


### 6.3 Discount Analysis

In [37]:
print("="*60)
print("DISCOUNT ANALYSIS")
print("="*60)

# Average discount
avg_discount = df_clean['Discount'].mean()
print(f"\nAverage Discount: {avg_discount:.2%}")

DISCOUNT ANALYSIS

Average Discount: 15.62%


In [38]:
# Discount distribution
print("\nDiscount Band Distribution:")
print("-"*60)
discount_distribution = df_clean['Discount_Band'].value_counts().sort_index()
print(discount_distribution)


Discount Band Distribution:
------------------------------------------------------------
Discount_Band
0%        4798
1-10%       94
11-20%    3709
21-30%     227
30%+      1166
Name: count, dtype: int64


In [39]:
# Discount vs Sales correlation
discount_sales_corr = df_clean['Discount'].corr(df_clean['Sales'])
print(f"\nCorrelation between Discount and Sales: {discount_sales_corr:.3f}")


Correlation between Discount and Sales: -0.028


In [40]:
# Discount vs Profit correlation
discount_profit_corr = df_clean['Discount'].corr(df_clean['Profit'])
print(f"Correlation between Discount and Profit: {discount_profit_corr:.3f}")

Correlation between Discount and Profit: -0.219


In [41]:
# Sales and Profit by Discount Band
print("\nSales and Profit by Discount Band:")
print("-"*60)
discount_analysis = df_clean.groupby('Discount_Band').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit_Margin': 'mean'
}).sort_index()
print(discount_analysis)


Sales and Profit by Discount Band:
------------------------------------------------------------
                   Sales     Profit  Profit_Margin
Discount_Band                                     
0%            1087908.47  320987.60          34.02
1-10%           54369.35    9029.18          15.58
11-20%         792152.89   91756.30          17.48
21-30%         103226.65  -10369.28         -11.55
30%+           259543.49 -125006.78         -91.47


### 6.4 Customer Analysis

In [42]:
print("="*60)
print("CUSTOMER ANALYSIS")
print("="*60)

# Total unique customers
total_customers = df_clean['Customer_ID'].nunique()
print(f"\nTotal Unique Customers: {total_customers:,}")

CUSTOMER ANALYSIS

Total Unique Customers: 793


In [43]:
# Revenue per customer
revenue_per_customer = total_sales / total_customers
print(f"Average Revenue per Customer: ${revenue_per_customer:,.2f}")

Average Revenue per Customer: $2,896.85


In [44]:
# Top 10 Customers by Sales
print("\nTop 10 Customers by Sales:")
print("-"*60)
top_customers = df_clean.groupby(['Customer_ID', 'Customer_Name'])['Sales'].sum().sort_values(ascending=False).head(10)
print(top_customers)


Top 10 Customers by Sales:
------------------------------------------------------------
Customer_ID  Customer_Name     
SM-20320     Sean Miller          25043.05
TC-20980     Tamara Chand         19052.22
RB-19360     Raymond Buch         15117.34
TA-21385     Tom Ashbrook         14595.62
AB-10105     Adrian Barton        14473.57
KL-16645     Ken Lonsdale         14175.23
SC-20095     Sanjit Chand         14142.33
HL-15040     Hunter Lopez         12873.30
SE-20110     Sanjit Engle         12209.44
CC-12370     Christopher Conant   12129.07
Name: Sales, dtype: float64


In [45]:
# Customer segment performance
print("\nCustomer Segment Performance:")
print("-"*60)
segment_performance = df_clean.groupby('Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Customer_ID': 'nunique'
}).rename(columns={'Customer_ID': 'Unique_Customers'})
segment_performance['Avg_Sales_per_Customer'] = segment_performance['Sales'] / segment_performance['Unique_Customers']
print(segment_performance)


Customer Segment Performance:
------------------------------------------------------------
                 Sales    Profit  Unique_Customers  Avg_Sales_per_Customer
Segment                                                                   
Consumer    1161401.34 134119.21               409                 2839.61
Corporate    706146.37  91979.13               236                 2992.15
Home Office  429653.15  60298.68               148                 2903.06


### 6.5 Product Analysis

In [46]:
print("="*60)
print("PRODUCT ANALYSIS")
print("="*60)

# Total unique products
total_products = df_clean['Product_ID'].nunique()
print(f"\nTotal Unique Products: {total_products:,}")

PRODUCT ANALYSIS

Total Unique Products: 1,862


In [47]:
# Top 10 Products by Sales
print("\nTop 10 Products by Sales:")
print("-"*60)
top_products_sales = df_clean.groupby('Product_Name')['Sales'].sum().sort_values(ascending=False).head(10)
print(top_products_sales)


Top 10 Products by Sales:
------------------------------------------------------------
Product_Name
Canon imageCLASS 2200 Advanced Copier                                         61599.82
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind   27453.38
Cisco TelePresence System EX90 Videoconferencing Unit                         22638.48
HON 5400 Series Task Chairs for Big and Tall                                  21870.58
GBC DocuBind TL300 Electric Binding System                                    19823.48
GBC Ibimaster 500 Manual ProClick Binding System                              19024.50
Hewlett Packard LaserJet 3310 Copier                                          18839.69
HP Designjet T520 Inkjet Large Format Printer - 24" Color                     18374.90
GBC DocuBind P400 Electric Binding System                                     17965.07
High Speed Automatic Electric Letter Opener                                   17030.31
Name: Sales, dtype: float64


In [48]:
# Sub-Category performance
print("\nSub-Category Performance (by Sales):")
print("-"*60)
subcategory_sales = df_clean.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False)
print(subcategory_sales)


Sub-Category Performance (by Sales):
------------------------------------------------------------
Sub-Category
Phones        330007.05
Chairs        328449.10
Storage       223843.61
Tables        206965.53
Binders       203412.73
Machines      189238.63
Accessories   167380.32
Copiers       149528.03
Bookcases     114880.00
Appliances    107532.16
Furnishings    91705.16
Paper          78479.21
Supplies       46673.54
Art            27118.79
Envelopes      16476.40
Labels         12486.31
Fasteners       3024.28
Name: Sales, dtype: float64


In [49]:
# Products with high sales but low profit (negative or low margin)
print("\nProducts with High Sales but Low Profit (Sales > $1000 and Profit Margin < 5%):")
print("-"*60)
high_sales_low_profit = df_clean[df_clean['Sales'] > 1000].groupby('Product_Name').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit_Margin': 'mean'
})
high_sales_low_profit = high_sales_low_profit[high_sales_low_profit['Profit_Margin'] < 5].sort_values('Sales', ascending=False).head(10)
print(high_sales_low_profit)


Products with High Sales but Low Profit (Sales > $1000 and Profit Margin < 5%):
------------------------------------------------------------
                                                      Sales   Profit  \
Product_Name                                                           
Cisco TelePresence System EX90 Videoconferencin... 22638.48 -1811.08   
HON 5400 Series Task Chairs for Big and Tall       20889.20   140.20   
GBC DocuBind P400 Electric Binding System          17965.07 -1878.17   
GBC Ibimaster 500 Manual ProClick Binding System   17806.93  2206.84   
High Speed Automatic Electric Letter Opener        17030.31  -262.00   
Lexmark MX611dhe Monochrome Laser Printer          16829.90 -4589.97   
Martin Yale Chadless Opener Electric Letter Opener 15989.95 -1149.28   
Ibico EPK-21 Electric Binding System               15875.92  3345.28   
Riverside Palais Royal Lawyers Bookcase, Royale... 15610.97  -669.54   
Cubify CubeX 3D Printer Double Head Print          11099.96 -8879.

---

## Section 7 — Advanced Business Analysis

### 7.1 Monthly Sales Trend

In [50]:
print("="*60)
print("MONTHLY SALES TREND")
print("="*60)

# Monthly sales and profit
monthly_trend = df_clean.groupby('Year_Month').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()
monthly_trend['Year_Month'] = monthly_trend['Year_Month'].astype(str)

print("\nMonthly Sales and Profit:")
print("-"*60)
print(monthly_trend.tail(12))

MONTHLY SALES TREND

Monthly Sales and Profit:
------------------------------------------------------------
   Year_Month     Sales   Profit
36    2017-01  43971.37  7140.44
37    2017-02  20301.13  1613.87
38    2017-03  58872.35 14751.89
39    2017-04  36521.54   933.29
40    2017-05  44261.11  6342.58
41    2017-06  52981.73  8223.34
42    2017-07  45264.42  6952.62
43    2017-08  63120.89  9040.96
44    2017-09  87866.65 10991.56
45    2017-10  77776.92  9275.28
46    2017-11 118447.82  9690.10
47    2017-12  83829.32  8483.35


In [51]:
# Visualization 1: Monthly Sales Trend
monthly_trend_plot = df_clean.groupby('Year_Month')['Sales'].sum().reset_index()
monthly_trend_plot['Year_Month'] = monthly_trend_plot['Year_Month'].astype(str)

fig = px.line(monthly_trend_plot, x='Year_Month', y='Sales', 
              title='Monthly Sales Trend',
              markers=True,
              labels={'Year_Month': 'Year-Month', 'Sales': 'Total Sales ($)'})
fig.update_layout(
    xaxis_title='Year-Month',
    yaxis_title='Total Sales ($)',
    hovermode='x unified'
)
fig.show()

In [52]:
# Visualization 2: Monthly Profit Trend
monthly_profit_plot = df_clean.groupby('Year_Month')['Profit'].sum().reset_index()
monthly_profit_plot['Year_Month'] = monthly_profit_plot['Year_Month'].astype(str)

fig = px.line(monthly_profit_plot, x='Year_Month', y='Profit', 
                title='Monthly Profit Trend',
                markers=True,
                labels={'Year_Month': 'Year-Month', 'Profit': 'Total Profit ($)'})
fig.update_layout(
    xaxis_title='Year-Month',
    yaxis_title='Total Profit ($)',
    hovermode='x unified'
)
fig.show()

In [53]:
# Visualization 3: Sales by Category
sales_by_category_df = sales_by_category.reset_index()
sales_by_category_df.columns = ['Category', 'Sales']

fig = px.bar(sales_by_category_df, x='Category', y='Sales',
              title='Sales by Category',
              labels={'Category': 'Category', 'Sales': 'Sales ($)'},
              color='Category')
fig.update_layout(
    xaxis_title='Category',
    yaxis_title='Sales ($)',
    showlegend=False
)
fig.show()

In [54]:
# Visualization 4: Profit by Category
profit_by_category_df = profit_by_category.reset_index()
profit_by_category_df.columns = ['Category', 'Profit']

fig = px.bar(profit_by_category_df, x='Category', y='Profit',
              title='Profit by Category',
              labels={'Category': 'Category', 'Profit': 'Profit ($)'},
              color='Category')
fig.update_layout(
    xaxis_title='Category',
    yaxis_title='Profit ($)',
    showlegend=False
)
fig.show()

In [55]:
# Visualization 5: Sales by Region
sales_by_region_df = sales_by_region.reset_index()
sales_by_region_df.columns = ['Region', 'Sales']

fig = px.bar(sales_by_region_df, x='Region', y='Sales',
              title='Sales by Region',
              labels={'Region': 'Region', 'Sales': 'Sales ($)'},
              color='Region')
fig.update_layout(
    xaxis_title='Region',
    yaxis_title='Sales ($)',
    showlegend=False
)
fig.show()

In [56]:
# Visualization 6: Profit by Region
profit_by_region_df = profit_by_region.reset_index()
profit_by_region_df.columns = ['Region', 'Profit']

fig = px.bar(profit_by_region_df, x='Region', y='Profit',
                title='Profit by Region',
                labels={'Region': 'Region', 'Profit': 'Profit ($)'},
                color='Region')
fig.update_layout(
    xaxis_title='Region',
    yaxis_title='Profit ($)',
    showlegend=False
)
fig.show()

In [57]:
# Visualization 7: Top 10 Products by Sales
top_products_sales_df = top_products_sales.reset_index()
top_products_sales_df.columns = ['Product_Name', 'Sales']

fig = px.bar(top_products_sales_df, x='Sales', y='Product_Name',
              title='Top 10 Products by Sales',
              labels={'Product_Name': 'Product Name', 'Sales': 'Sales ($)'},
              orientation='h')
fig.update_layout(
    xaxis_title='Sales ($)',
    yaxis_title='Product Name'
)
fig.show()

In [58]:
# Visualization 8: Top 10 Products by Profit
top_products_profit_df = top_products_profit.reset_index()
top_products_profit_df.columns = ['Product_Name', 'Profit']

fig = px.bar(top_products_profit_df, x='Profit', y='Product_Name',
              title='Top 10 Products by Profit',
              labels={'Product_Name': 'Product Name', 'Profit': 'Profit ($)'},
              orientation='h')
fig.update_layout(
    xaxis_title='Profit ($)',
    yaxis_title='Product Name'
)
fig.show()

In [59]:
# Visualization 9: Bottom 10 Products by Profit
bottom_products_profit_df = bottom_products_profit.reset_index()
bottom_products_profit_df.columns = ['Product_Name', 'Profit']

fig = px.bar(bottom_products_profit_df, x='Profit', y='Product_Name',
                title='Bottom 10 Products by Profit (Loss-Making)',
                labels={'Product_Name': 'Product Name', 'Profit': 'Profit ($)'},
                orientation='h')
fig.update_layout(
    xaxis_title='Profit ($)',
    yaxis_title='Product Name'
)
fig.show()

In [60]:
# Visualization 10: Discount vs Profit Scatter Plot
fig = px.scatter(df_clean, x='Discount', y='Profit',
                 title='Discount vs Profit',
                 labels={'Discount': 'Discount', 'Profit': 'Profit ($)'},
                 opacity=0.5)
fig.update_layout(
    xaxis_title='Discount',
    yaxis_title='Profit ($)'
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.show()

In [61]:
# Visualization 11: Sales vs Profit Scatter Plot
fig = px.scatter(df_clean, x='Sales', y='Profit',
                title='Sales vs Profit',
                labels={'Sales': 'Sales ($)', 'Profit': 'Profit ($)'},
                opacity=0.5)
fig.update_layout(
    xaxis_title='Sales ($)',
    yaxis_title='Profit ($)'
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.show()

In [62]:
# Visualization 12: Customer Segment Comparison
segment_comparison = df_clean.groupby('Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()

fig = px.bar(segment_comparison, x='Segment', y=['Sales', 'Profit'],
              title='Customer Segment Comparison',
              labels={'Segment': 'Segment', 'value': 'Amount ($)', 'variable': 'Metric'},
              barmode='group')
fig.update_layout(
    xaxis_title='Segment',
    yaxis_title='Amount ($)',
    legend_title='Metric'
)
fig.show()

In [63]:
# Visualization 13: State-Level Performance (Top 15)
state_performance = df_clean.groupby('State')['Sales'].sum().sort_values(ascending=False)
top_15_states = state_performance.head(15).reset_index()
top_15_states.columns = ['State', 'Sales']

fig = px.bar(top_15_states, x='Sales', y='State',
                title='Top 15 States by Sales',
                labels={'State': 'State', 'Sales': 'Sales ($)'},
                orientation='h')
fig.update_layout(
    xaxis_title='Sales ($)',
    yaxis_title='State'
)
fig.show()

In [64]:
# Visualization 14: Profit Margin by Category
category_margin = df_clean.groupby('Category')['Profit_Margin'].mean()
category_margin_df = category_margin.reset_index()

fig = px.bar(category_margin_df, x='Category', y='Profit_Margin',
            title='Profit Margin by Category',
            labels={'Category': 'Category', 'Profit_Margin': 'Profit Margin (%)'},
            color='Category')
fig.update_layout(
    xaxis_title='Category',
    yaxis_title='Profit Margin (%)',
    showlegend=False
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.show()

In [65]:
# Visualization 15: Pareto Chart - Customer Contribution
customer_sales = df_clean.groupby('Customer_ID')['Sales'].sum().sort_values(ascending=False)
customer_sales_df = customer_sales.reset_index()
customer_sales_df.columns = ['Customer_ID', 'Sales']
customer_sales_df['Cumulative_Sales'] = customer_sales_df['Sales'].cumsum()
customer_sales_df['Cumulative_Percentage'] = (customer_sales_df['Cumulative_Sales'] / customer_sales_df['Sales'].sum()) * 100
customer_sales_df['Customer_Rank'] = range(1, len(customer_sales_df) + 1)

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar chart - customer sales
fig.add_trace(
    go.Bar(x=customer_sales_df['Customer_Rank'], y=customer_sales_df['Sales'],
            name='Sales', marker_color='#1f77b4', opacity=0.7),
    secondary_y=False,
)

# Line chart - cumulative percentage
fig.add_trace(
    go.Scatter(x=customer_sales_df['Customer_Rank'], y=customer_sales_df['Cumulative_Percentage'],
                name='Cumulative %', marker_color='#d62728', line=dict(width=2)),
    secondary_y=True,
)

# Add 80% reference line
fig.add_hline(y=80, line_dash='dash', line_color='green', secondary_y=True,
                annotation_text='80%', annotation_position='right')

fig.update_xaxes(title_text='Customer Rank')
fig.update_yaxes(title_text='Sales ($)', secondary_y=False)
fig.update_yaxes(title_text='Cumulative Percentage (%)', secondary_y=True)
fig.update_layout(title_text='Pareto Chart - Customer Contribution to Sales',
                    hovermode='x unified')
fig.show()

---

## Section 9 — Save Cleaned Dataset

In [66]:
# Save cleaned dataset
df_clean.to_csv('../data/processed/ecommerce_clean.csv', index=False)
print("Cleaned dataset saved to: ../data/processed/ecommerce_clean.csv")
print(f"Final dataset shape: {df_clean.shape}")

Cleaned dataset saved to: ../data/processed/ecommerce_clean.csv
Final dataset shape: (9994, 32)


---

## Section 10 — Key KPIs Summary

In [67]:
print("="*60)
print("KEY KPIS SUMMARY")
print("="*60)

# Calculate key KPIs
total_orders = df_clean['Order_ID'].nunique()
total_quantity = df_clean['Quantity'].sum()
avg_order_value = total_sales / total_orders

kpis = pd.DataFrame({
    'KPI': [
        'Total Sales',
        'Total Profit',
        'Profit Margin',
        'Total Orders',
        'Total Customers',
        'Total Quantity',
        'Average Order Value',
        'Average Discount'
    ],
    'Value': [
        f'${total_sales:,.2f}',
        f'${total_profit:,.2f}',
        f'{overall_profit_margin:.2f}%',
        f'{total_orders:,}',
        f'{total_customers:,}',
        f'{total_quantity:,}',
        f'${avg_order_value:,.2f}',
        f'{avg_discount:.2%}'
    ]
})

print(kpis.to_string(index=False))

KEY KPIS SUMMARY
                KPI         Value
        Total Sales $2,297,200.86
       Total Profit   $286,397.02
      Profit Margin        12.47%
       Total Orders         5,009
    Total Customers           793
     Total Quantity        37,873
Average Order Value       $458.61
   Average Discount        15.62%


---

## Conclusion

This comprehensive E-commerce Sales & Profit Analysis has covered:

1. **Data Quality Assessment** - Identified missing values, duplicates, and data quality issues
2. **Data Cleaning** - Standardized column names, converted dates, created derived features
3. **Sales Analysis** - Analyzed sales across time, categories, regions, and segments
4. **Profit Analysis** - Identified profitability patterns and loss-making transactions
5. **Discount Analysis** - Evaluated the impact of discounts on sales and profit
6. **Customer Analysis** - Segmented customers and identified top performers
7. **Product Analysis** - Identified top and bottom performing products
8. **Advanced Analysis** - Monthly trends, profitability matrices, regional analysis, Pareto analysis
9. **Visualizations** - Created 15 professional charts for business insights

The cleaned dataset has been saved for use in SQL, Power BI, and Excel analysis.

**Next Steps:**
- SQL database setup and analysis
- Power BI dashboard development
- Excel analysis and dashboard creation
- Business insights and recommendations documentation

In [69]:
# Discount vs Profitability Analysis

discount_analysis = (
    df.assign(
        Discount_Band=pd.cut(
            df["Discount"],
            bins=[-0.01, 0, 0.10, 0.20, 0.30, 0.40, 1],
            labels=["0%", "1-10%", "11-20%", "21-30%", "31-40%", "40%+"]
        )
    )
    .groupby("Discount_Band", observed=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Discount=("Discount", "mean"),
        Orders=("Order ID", "nunique")
    )
    .reset_index()
)

discount_analysis["Profit_Margin"] = (
    discount_analysis["Total_Profit"]
    / discount_analysis["Total_Sales"]
    * 100
)

discount_analysis

,Discount_Band,Total_Sales,Total_Profit,Average_Discount,Orders,Profit_Margin
0,0%,1087908.47,320987.60,0.00,2644,29.51
1,1-10%,54369.35,9029.18,0.10,89,16.61
2,11-20%,792152.89,91756.30,0.20,2436,11.58
3,21-30%,103226.65,-10369.28,0.30,211,-10.05
4,31-40%,130911.24,-25448.19,0.39,211,-19.44
5,40%+,128632.25,-99558.59,0.70,737,-77.40


In [70]:
# Profit Margin by Discount Band

fig = px.bar(
    discount_analysis,
    x="Discount_Band",
    y="Profit_Margin",
    title="Profit Margin by Discount Band",
    labels={
        "Discount_Band": "Discount Band",
        "Profit_Margin": "Profit Margin (%)"
    },
    text="Profit_Margin"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    annotation_text="Break-even"
)

fig.update_layout(
    template="plotly_white",
    height=500
)

fig.show()

### Business Insight — Discount & Profitability

The analysis shows a strong inverse relationship between discount intensity and profitability.

- Orders with no discount generate the highest profit margin of 29.5%.
- Profitability declines progressively as discount levels increase.
- Discount bands above 20% result in negative profit margins.
- The 40%+ discount band has the largest negative margin at -77.4%.

**Business implication:** Higher discounts should be used selectively because aggressive discounting can increase sales activity while significantly reducing or eliminating profitability.

In [71]:
# Customer Segment Profitability Analysis

segment_analysis = (
    df.groupby("Segment")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique"),
          Average_Discount=("Discount", "mean")
      )
      .reset_index()
)

segment_analysis["Profit_Margin"] = (
    segment_analysis["Total_Profit"] /
    segment_analysis["Total_Sales"] * 100
)

segment_analysis = segment_analysis.sort_values(
    "Total_Profit", ascending=False
)

segment_analysis

,Segment,Total_Sales,Total_Profit,Total_Orders,Average_Discount,Profit_Margin
0,Consumer,1161401.34,134119.21,2586,0.16,11.55
1,Corporate,706146.37,91979.13,1514,0.16,13.03
2,Home Office,429653.15,60298.68,909,0.15,14.03


In [72]:
# Profit Margin by Customer Segment

import plotly.express as px

fig = px.bar(
    segment_analysis,
    x="Segment",
    y="Profit_Margin",
    text="Profit_Margin",
    title="Profit Margin by Customer Segment",
    labels={
        "Profit_Margin": "Profit Margin (%)",
        "Segment": "Customer Segment"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    yaxis_title="Profit Margin (%)",
    xaxis_title="Customer Segment"
)

fig.show()

### Business Insight — Customer Segment Profitability

Customer segment performance shows that sales volume and profitability are not directly proportional.

- The **Consumer** segment generates the highest sales ($1.16M) and total profit ($134.1K).
- The **Home Office** segment has the highest profit margin at **14.03%**, despite having the lowest sales among the three segments.
- The **Corporate** segment achieves a **13.03%** profit margin, higher than the Consumer segment's 11.55%.

**Business implication:** Customer segments should be evaluated using both revenue and profitability. A segment with lower sales can still generate stronger margins and may offer opportunities for targeted, higher-value strategies.

In [74]:
# Sub-Category Profitability Analysis

subcategory_analysis = (
    df.groupby("Sub-Category")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Orders=("Order ID", "nunique"),
          Average_Discount=("Discount", "mean")
      )
      .reset_index()
)

subcategory_analysis["Profit_Margin"] = (
    subcategory_analysis["Total_Profit"] /
    subcategory_analysis["Total_Sales"] * 100
)

subcategory_analysis = subcategory_analysis.sort_values(
    "Total_Profit", ascending=False
)

subcategory_analysis

,Sub-Category,Total_Sales,Total_Profit,Total_Orders,Average_Discount,Profit_Margin
6,Copiers,149528.03,55617.82,68,0.16,37.20
13,Phones,330007.05,44515.73,814,0.15,13.49
0,Accessories,167380.32,41936.64,718,0.08,25.05
12,Paper,78479.21,34053.57,1191,0.07,43.39
3,Binders,203412.73,30221.76,1316,0.37,14.86
5,Chairs,328449.10,26590.17,576,0.17,8.10
14,Storage,223843.61,21278.83,777,0.07,9.51
1,Appliances,107532.16,18138.01,451,0.17,16.87
9,Furnishings,91705.16,13059.14,877,0.14,14.24
7,Envelopes,16476.40,6964.18,249,0.08,42.27


In [75]:
# Profitability by Sub-Category

import plotly.express as px

subcategory_chart = subcategory_analysis.sort_values(
    "Profit_Margin",
    ascending=True
)

fig = px.bar(
    subcategory_chart,
    x="Profit_Margin",
    y="Sub-Category",
    orientation="h",
    text="Profit_Margin",
    title="Profit Margin by Product Sub-Category",
    labels={
        "Profit_Margin": "Profit Margin (%)",
        "Sub-Category": "Product Sub-Category"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.add_vline(
    x=0,
    line_dash="dash",
    annotation_text="Break-even"
)

fig.update_layout(
    xaxis_title="Profit Margin (%)",
    yaxis_title="Product Sub-Category"
)

fig.show()

### Business Insight — Sub-Category Profitability

Sub-category performance shows significant differences in profitability across products.

- **Tables** have the weakest performance, generating a negative profit margin of **-8.56%**.
- **Bookcases** and **Supplies** also operate at negative profit margins of **-3.02%** and **-2.55%**, respectively.
- **Machines** generate relatively high sales of approximately **$189K**, but have a low profit margin of only **1.79%**.
- **Paper** and **Copiers** show strong profitability, with profit margins of **43.39%** and **37.20%**, respectively.

**Business implication:** Product-level profitability should be considered alongside sales volume. High-revenue products are not necessarily highly profitable, while some lower-sales sub-categories can generate stronger margins.